# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://mlcommons.github.io/croissant/) library for loading, exploring, and processing the FAIR² dataset package.

### Dataset Source
The dataset Croissant schema is provided via this URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via schema URL
dataset = mlc.Dataset(croissant_url)

# Print the dataset metadata: name and description only
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by @id

record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in the schema. Trying to extract file-based record sets...")
    # Fallback: Try to enumerate DataFiles
    # For mlcroissant 1.0 this is dataset.metadata.distribution (usually data files or assets)
    if hasattr(dataset.metadata, 'distributions'):
        datafiles = dataset.metadata.distributions
    elif hasattr(dataset.metadata, 'distribution'):
        datafiles = dataset.metadata.distribution
    else:
        datafiles = []
    print(f"Number of file distributions: {len(datafiles)}\n")
    for i, dfile in enumerate(datafiles):
        print(f"[{i+1}] Distribution @id: {getattr(dfile, '@id', getattr(dfile, 'id', None))}")
        print(f"    - Name: {getattr(dfile, 'name', None)}")
        print(f"    - Content URL: {getattr(dfile, 'content_url', None)}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for f in fields:
                field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
                print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, the record sets are not listed directly in 'record_sets'.
# Inspection in cell 6 usually shows file-level @id.
# According to the Croissant schema, try to load from first datafile distribution.

# List all distributions: These are potential record_set targets
if hasattr(dataset.metadata, 'distribution'):
    distributions = dataset.metadata.distribution
elif hasattr(dataset.metadata, 'distributions'):
    distributions = dataset.metadata.distributions
else:
    distributions = []

# Choose the first distribution
assert len(distributions) > 0, "No distribution found in the metadata."
record_set_id = distributions[0]['@id']
print(f"Selected record set @id: {record_set_id}")

# List all record sets for further extraction
record_set_ids = [d['@id'] for d in distributions]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"No records found in record set {rs_id}, skipping.")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for {rs_id} with columns: {df.columns.tolist()}")

# Display first few rows of the primary record set if loaded
if record_set_id in dataframes:
    print(dataframes[record_set_id].head())
else:
    print("Primary record set could not be extracted as DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we select a numeric field, filter, normalize and group as example.

In [ ]:
# Check if we have the dataframe
if record_set_id in dataframes:
    df = dataframes[record_set_id]
    # List all numeric-like columns
    numeric_like = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_like:
        # Try to infer numeric columns by trying to convert object columns
        numeric_like = []
        for c in df.columns:
            try:
                _ = pd.to_numeric(df[c].dropna().iloc[0])
                numeric_like.append(c)
            except Exception:
                continue
    if numeric_like:
        numeric_field = numeric_like[0]
        print(f"Using numeric field: {numeric_field}")
        try:
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        except Exception as e:
            print(e)
        threshold = df[numeric_field].mean() if not df[numeric_field].isna().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first non-numeric column
        non_numeric = [c for c in df.columns if c not in numeric_like]
        if non_numeric:
            group_field = non_numeric[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped data by {group_field} (mean of {numeric_field}):")
                print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field if available
if record_set_id in dataframes and 'numeric_field' in locals() and numeric_field in dataframes[record_set_id].columns:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# If at least 2 numeric fields, do scatterplot
if record_set_id in dataframes and 'numeric_like' in locals() and len(numeric_like) > 1:
    plt.figure(figsize=(6,5))
    sns.scatterplot(x=numeric_like[0], y=numeric_like[1], data=dataframes[record_set_id])
    plt.title(f'Scatterplot of {numeric_like[0]} vs {numeric_like[1]}')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access, load and explore the [FAIR² mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa), referencing all entities by their Croissant `@id` field using `mlcroissant`.

- The dataset's metadata can be inspected programmatically, including all record/file set `@id`s, fields, and columns.
- We loaded tabular data from the main dataset file and performed basic EDA operations: numeric filtering, normalization, and grouping.
- Basic visualizations showed the distribution of a primary numeric field.

Further steps may involve domain-specific analysis using the detailed protein attributes in this dataset.